# 06 - Agent Memory

## Scenario: Cross-Session Context

If a customer emails support on Monday, and then emails again on Friday, a standard agent forgets the Monday conversation. 

Advanced agents use **Memory Systems** to retrieve past context.
- **Short-Term Memory**: The `messages` array in the current session (context window).
- **Long-Term Memory**: A Vector Database (like ChromaDB or Pinecone) storing past conversations or resolved incident reports.

In this notebook, we will give our Northstar Agent long-term memory of a past outage.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. Setting up Long-Term Memory (Vector DB)

In [2]:
import chromadb

# We use an in-memory Chroma database for demonstration
chroma_client = chromadb.Client()
memory_collection = chroma_client.create_collection(name="past_incidents")

# We "remember" a past incident
memory_collection.add(
    documents=[
        "INC-101 (Last Month): The Redis cache for the checkout service OOM'd. We fixed it by flushing the cache and increasing the memory limit to 4GB."
    ],
    ids=["inc_101"]
)
print("🧠 Long-term memory initialized.")


🧠 Long-term memory initialized.


## 2. Augmented Investigation

When a new incident occurs, the agent queries its memory *before* acting.

In [3]:
def handle_new_incident(description: str):
    print(f"🚨 New Incident: {description}")
    
    # 1. Query Long-Term Memory
    print("🔍 [Agent] Searching memory for similar past incidents...")
    results = memory_collection.query(query_texts=[description], n_results=1)
    
    past_context = ""
    if results['documents'] and results['documents'][0]:
        past_context = results['documents'][0][0]
        print(f"💡 [Agent] Found relevant memory: {past_context}")
    else:
        print("💡 [Agent] No relevant memories found.")
        
    # 2. Construct Augmented Prompt
    prompt = f"""
    You are an SRE resolving a new incident: '{description}'.
    Here is a memory of a similar past incident: '{past_context}'.
    Use the past memory to suggest a fix immediately.
    """
    
    # 3. Execute Agent (Mocked output)
    print("\n🤖 [Agent Output] Based on INC-101, I highly recommend flushing the Redis cache immediately and verifying the memory limit is set to 4GB.")

handle_new_incident("Checkout service is timing out, looks like a cache issue.")


🚨 New Incident: Checkout service is timing out, looks like a cache issue.
🔍 [Agent] Searching memory for similar past incidents...
💡 [Agent] Found relevant memory: INC-101 (Last Month): The Redis cache for the checkout service OOM'd. We fixed it by flushing the cache and increasing the memory limit to 4GB.

🤖 [Agent Output] Based on INC-101, I highly recommend flushing the Redis cache immediately and verifying the memory limit is set to 4GB.


## Checkpoint

**1. What is the difference between Short-Term and Long-Term memory in an LLM Agent?**
- A) Short-term is fast, Long-term is slow.
- B) Short-term is the current prompt's `messages` array (bounded by token limits). Long-term relies on external storage (like a Vector DB) to retrieve relevant context across separate sessions.
- C) Short-term uses Python, Long-term uses SQL.
- D) Only human agents have Long-Term memory.
